# 12 作りたい材料像から始める逆解析

逆解析の入力は「すべての物性を必須合格にする条件」ではなく、**何を作りたいか**です。

このNotebookでは、用途イメージを選ぶと、その用途に関係する物性の組合せを表示し、主目標を優先しながら副目標とのトレードオフを持つ候補を探索します。副目標を同時にすべて満たすことは要求しません。

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(project_root / 'src'))

import numpy as np
import pandas as pd
from IPython.display import display

from plastech.inverse_design import (
    TargetSpecification,
    build_candidate_library,
    candidate_summary,
    fit_forward_models,
    generate_interpolated_candidates,
    specifications_frame,
)
from plastech.modeling import column_metadata
from plastech.target_selection import cross_validated_forward_predictions


## 作りたい材料像を選ぶ

`PRODUCT_CONCEPT` を変更すると、対応する物性の組合せに切り替わります。数値はデータ分布を踏まえた探索用の初期案です。実際の製品仕様・規格がある場合は、その値に置き換えます。

- 最初の物性が**主目標**です。候補生成の起点と順位付けで最も重視します。
- 以降の物性は**副目標**です。良いほど望ましいものとして評価しますが、満たさないだけで候補を除外しません。

In [2]:
PRODUCT_CONCEPT = '柔軟・低温用途'  # ここを変更する

DESIGN_BRIEFS = {
    '柔軟・低温用途': {
        'description': '柔らかく、低温で脆くなりにくい材料を狙う。例: 軟質シート、低温用途のホース。',
        'preferences': (
            TargetSpecification('S03010_030010', '硬度 HDA (0s)', 70.0, 4.0, weight=1.4),
            TargetSpecification('S06060_060030', '引張伸び (JIS2 200mm/min)', 370.0, 40.0, objective='lower_bound', weight=1.0),
            TargetSpecification('S13010_130010', '脆化温度 (JIS K6723)', -41.0, 4.0, objective='upper_bound', weight=1.0),
            TargetSpecification('S01010_010010', '比重 (23℃)', 1.34, 0.06, weight=0.5),
        ),
    },
    '硬さ・剛性用途': {
        'description': '硬度と剛性を優先する材料を狙う。例: 比較的硬いシート、剛性が必要な成形材。',
        'preferences': (
            TargetSpecification('S03010_030010', '硬度 HDA (0s)', 89.0, 3.0, weight=1.4),
            TargetSpecification('S06010_060030', '100%モジュラス (JIS2 200mm/min)', 11.0, 2.0, objective='lower_bound', weight=1.0),
            TargetSpecification('S06020_060030', '引張強さ (JIS2 200mm/min)', 20.0, 3.0, objective='lower_bound', weight=0.8),
            TargetSpecification('S01010_010010', '比重 (23℃)', 1.38, 0.06, weight=0.4),
        ),
    },
    'バランス型軟質PVC': {
        'description': '硬度、剛性、低温特性、比重のバランスを狙う。',
        'preferences': (
            TargetSpecification('S03010_030010', '硬度 HDA (0s)', 78.0, 3.0, weight=1.3),
            TargetSpecification('S06010_060030', '100%モジュラス (JIS2 200mm/min)', 7.5, 1.5, weight=1.0),
            TargetSpecification('S13010_130010', '脆化温度 (JIS K6723)', -33.0, 3.0, objective='upper_bound', weight=1.0),
            TargetSpecification('S01010_010010', '比重 (23℃)', 1.34, 0.05, weight=0.5),
        ),
    },
}

if PRODUCT_CONCEPT not in DESIGN_BRIEFS:
    raise KeyError(f'Unknown PRODUCT_CONCEPT: {PRODUCT_CONCEPT}. Available: {list(DESIGN_BRIEFS)}')

brief = DESIGN_BRIEFS[PRODUCT_CONCEPT]
preferences = brief['preferences']
main_goal = preferences[0]

print('作りたいもの:', PRODUCT_CONCEPT)
print('説明:', brief['description'])
print('主目標:', main_goal.name, main_goal.description())
display(specifications_frame(preferences))


作りたいもの: 柔軟・低温用途
説明: 柔らかく、低温で脆くなりにくい材料を狙う。例: 軟質シート、低温用途のホース。
主目標: 硬度 HDA (0s) 70 ± 4


,column,property,objective,target,tolerance,weight,requirement
0,S03010_030010,硬度 HDA (0s),target,70.00,4.00,1.4,70 ± 4
1,S06060_060030,引張伸び (JIS2 200mm/min),lower_bound,370.00,40.00,1.0,>= 370
2,S13010_130010,脆化温度 (JIS K6723),upper_bound,-41.00,4.00,1.0,<= -41
3,S01010_010010,比重 (23℃),target,1.34,0.06,0.5,1.34 ± 0.06


## 配合から物性を予測するLightGBMモデルを学習する

各物性に順解析モデルを作ります。逆解析では、生成した候補配合をこのモデルに入力し、物性ごとの予測値を得ます。

In [3]:
forward_models = fit_forward_models(
    preferences,
    dataset_name='processed_main',
    model_name='lightgbm',
    n_estimators=800,
    random_state=42,
)


FileNotFoundError: config/data_catalog.local.json がありません。config/data_catalog.example.json を参考にローカル用の対応表を作成してください。

## 主目標に近い実配合を起点に仮想候補を作る

主目標の実測値が近い既存配合を30件選びます。同一の主PVC・主可塑剤系列にある処方を補間して候補を作るため、無制限な新規原料の組合せは作りません。

In [ ]:
def preference_error(values, specification):
    if specification.objective == 'target':
        return (values - specification.target).abs() / specification.tolerance
    if specification.objective == 'lower_bound':
        return (specification.target - values).clip(lower=0) / specification.tolerance
    if specification.objective == 'upper_bound':
        return (values - specification.target).clip(lower=0) / specification.tolerance
    raise ValueError(f'Unknown objective: {specification.objective}')

library = build_candidate_library(forward_models, dataset_name='processed_main')
main_observed = pd.to_numeric(library[f'observed__{main_goal.column}'], errors='coerce')
seed_ranking = library.copy()
seed_ranking['main_goal_error'] = preference_error(main_observed, main_goal)
seed_ranking = seed_ranking.sort_values('main_goal_error', kind='stable')

virtual_candidates = generate_interpolated_candidates(
    library,
    seed_ranking,
    forward_models,
    seed_count=30,
    partners_per_seed=5,
    fractions=(0.25, 0.5, 0.75),
    random_state=42,
)
print(f'主目標を実測済みの候補ライブラリ: {len(library):,} 件')
print(f'生成した仮想候補: {len(virtual_candidates):,} 件')


## Pareto最適候補を残す

各副目標の良し悪しを誤差として表し、他の候補より全物性で劣る候補だけを落とします。残る候補は、どれか1つの物性で優れているため、トレードオフを比較する価値があります。

`design_score` は表示順を決めるための重み付き指標です。唯一の正解を意味しません。

In [ ]:
def pareto_optimal_mask(error_matrix):
    # 小さいほど良い誤差行列に対し、支配されない候補を残す。
    mask = np.ones(len(error_matrix), dtype=bool)
    for i, error in enumerate(error_matrix):
        dominates_i = np.all(error_matrix <= error, axis=1) & np.any(error_matrix < error, axis=1)
        if dominates_i.any():
            mask[i] = False
    return mask

scored_candidates = virtual_candidates.copy()
error_columns = []
for specification in preferences:
    column = f'preference_error__{specification.column}'
    values = scored_candidates[f'predicted__{specification.column}']
    scored_candidates[column] = preference_error(values, specification)
    error_columns.append(column)

total_weight = sum(specification.weight for specification in preferences)
weighted_error = sum(
    specification.weight * scored_candidates[f'preference_error__{specification.column}']
    for specification in preferences
)
scored_candidates['design_score'] = weighted_error / total_weight
scored_candidates['pareto_optimal'] = pareto_optimal_mask(
    scored_candidates[error_columns].to_numpy()
)
pareto_candidates = scored_candidates[scored_candidates['pareto_optimal']].copy()
pareto_candidates['material_series'] = (
    pareto_candidates['primary_pvc'] + ' | ' + pareto_candidates['primary_plasticizer']
)

# 異なる材料系列を優先して、実験候補を最大5件提示する。
recommended_candidates = (
    pareto_candidates.sort_values('design_score')
    .drop_duplicates('material_series')
    .head(5)
    .reset_index(drop=True)
)

print(f'Pareto最適候補: {len(pareto_candidates):,} 件')
print(f'異なる材料系列として提示する候補: {len(recommended_candidates):,} 件')
display(candidate_summary(recommended_candidates, forward_models, top_n=5))
display(recommended_candidates[['design_score', *error_columns]].head(5))


## 予測誤差を確認する

候補の予測値は目標に近くても、実測には誤差があります。OOF誤差を確認し、順位の差を過信しないようにします。

In [ ]:
uncertainty_rows = []
for specification in preferences:
    cv_result = cross_validated_forward_predictions(
        specification.column,
        dataset_name='processed_main',
        model_name='lightgbm',
        n_splits=5,
        n_estimators=800,
        random_state=42,
    )
    metric = cv_result.metrics.iloc[0]
    uncertainty_rows.append({
        'property': specification.name,
        'role': '主目標' if specification == main_goal else '副目標',
        'preference': specification.description(),
        'OOF_MAE': metric['MAE'],
        'OOF_p90_abs_error': metric['p90_abs_error'],
    })
display(pd.DataFrame(uncertainty_rows))


## 第1候補の処方を親配合と比較する

第1候補は、親配合AとBの補間です。原料量、原料併用可否、総配合量、加工性、コスト、法規は、実験前に配合担当者が確認します。

In [ ]:
selected = recommended_candidates.iloc[0]
parent_a = library.set_index('row_id').loc[selected['parent_a']]
parent_b = library.set_index('row_id').loc[selected['parent_b']]
feature_columns = sorted({
    column for prepared in forward_models.prepared.values() for column in prepared.feature_columns
})
metadata = column_metadata('processed_main').set_index('column')

recipe_comparison = pd.DataFrame({
    'column': feature_columns,
    'display_name': [metadata.at[column, 'display_name'] for column in feature_columns],
    'parent_a': parent_a[feature_columns].astype(float).values,
    'parent_b': parent_b[feature_columns].astype(float).values,
    'virtual_candidate': selected[feature_columns].astype(float).values,
})
recipe_comparison = recipe_comparison[
    recipe_comparison[['parent_a', 'parent_b', 'virtual_candidate']].ne(0).any(axis=1)
].sort_values('virtual_candidate', ascending=False)

print('候補ID:', selected['row_id'])
print('親配合A:', selected['parent_a'])
print('親配合B:', selected['parent_b'])
print('親配合Aの比率:', selected['parent_a_fraction'])
display(recipe_comparison)


## 解釈

- このNotebookは「唯一の正解配合」を出すものではない。
- 用途に応じた主目標・副目標のトレードオフを持つ実験候補を提示する。
- 次に実験する候補は、予測値、OOF誤差、原料・加工・コスト条件をまとめて人が選ぶ。
- 用途の要求が明確になったら、`DESIGN_BRIEFS` に新しい材料像と物性の組合せを追加する。